In [1]:
import torchvision.models as models

# Print the names of all segmentation models in torchvision
segmentation_models = [name for name in dir(models.segmentation) if not name.startswith("__")]
print('deeplabv3_resnet50' in segmentation_models)

True


In [2]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

# Path to the directories
root = './'
image_dir = 'leftImg8bit_trainvaltest/leftImg8bit/train/aachen'  # Update this path
mask_dir = 'gtFine_trainvaltest/gtFine/train/aachen'               # Update this path

# Transformations (if any)
transform = transforms.Compose([
    transforms.ToTensor(),
    # Add any other transformations you need
])

# Custom dataset class
class CustomDataset(datasets.ImageFolder):
    def __init__(self, root,img_dir, mask_dir, transform=None):
        self.root = root
        self.img_dir = os.path.join(root,img_dir)
        self.mask_dir = os.path.join(root,mask_dir)
        self.transform = transform
        self.images = os.listdir(img_dir)

    def __getitem__(self, index):
        img_path = os.path.join(self.img_dir, self.images[index])
        mask_path = os.path.join(self.mask_dir, self.images[index].replace('_leftImg8bit.png', '_gtFine_labelIds.png'))  # Adjust mask file naming as needed
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Assuming mask is grayscale

        if self.transform is not None:
            image = self.transform(image)
            mask = self.transform(mask)

        return {'image':image, 'mask':mask}

    def __len__(self):
        return len(self.images)

# Create the dataset and data loader
dataset = CustomDataset(root,image_dir, mask_dir, transform)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)  # Adjust batch_size and shuffle as needed

In [3]:
for batch in data_loader:
    images = batch['image']
    masks = batch['mask']

In [4]:
print(masks.shape,images.shape)

torch.Size([2, 1, 1024, 2048]) torch.Size([2, 3, 1024, 2048])


In [5]:
import torch
import torchvision.models as models
import torchvision.models.segmentation as segmentation

# Load pre-trained ResNet-34
resnet34 = models.resnet34(pretrained=True)

# Create a DeepLabv3+ model with ResNet-34 as the backbone
deeplab_model = segmentation.deeplabv3_resnet50(pretrained=False, num_classes=30)

/Users/anubhavsingh/opt/anaconda3/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/anubhavsingh/opt/anaconda3/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/Users/anubhavsingh/opt/anaconda3/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [9]:
output = deeplab_model(images)

In [6]:
import torch.nn as nn
import torch.optim as optim

num_classes = 2  # Assuming binary segmentation (sidewalk or not)
batch_size = 4
learning_rate = 0.1
num_epochs = 1

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(deeplab_model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    deeplab_model.train()
    for batch in data_loader:
        images = batch['image']
        masks = batch['mask']
        
        optimizer.zero_grad()
        outputs = deeplab_model(images)
        
        masks = masks.squeeze(1)  # Assuming masks have shape (batch_size, 1, height, width)

        # Ensure that masks are of type Long (for class indices)
        masks = masks.long()
        
        loss = criterion(outputs['out'], masks)  # Assuming 'out' is the output for segmentation
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}')

# Save the trained model
torch.save(deeplab_model.state_dict(), 'deeplab_model.pth')

KeyboardInterrupt: 

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
import torch.nn as nn
import torch.optim as optim

num_classes = 2  # Assuming binary segmentation (sidewalk or not)
batch_size = 4
learning_rate = 0.1
num_epochs = 1

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(deeplab_model.parameters(), lr=learning_rate)

# Training loop

deeplab_model.train()
for batch in data_loader:
    images = batch['image']
    masks = batch['mask']
        
    optimizer.zero_grad()
    outputs = deeplab_model(images)
        
    masks = masks.squeeze(1)  # Assuming masks have shape (batch_size, 1, height, width)

        # Ensure that masks are of type Long (for class indices)
    masks = masks.long()
        
    loss = criterion(outputs['out'], masks)  # Assuming 'out' is the output for segmentation
    loss.backward()
    optimizer.step()

print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}')

# Save the trained model
torch.save(deeplab_model.state_dict(), 'deeplab_model.pth')

KeyboardInterrupt: 